In [1]:
# !MAX_JOBS=4 TORCH_CUDA_ARCH_LIST="7.5" CAUSAL_CONV1D_FORCE_BUILD=TRUE \
#   pip wheel causal-conv1d --no-build-isolation --no-deps -w /kaggle/working/wheels

In [2]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [3]:
import glob
whl = glob.glob("/kaggle/input/**/causal_conv1d-*.whl", recursive=True)
print(whl)

['/kaggle/input/datasets/erygfry/cached-data/causal_conv1d-1.7.0-cp313-cp313-linux_x86_64.whl']


In [4]:
!pip install --no-deps {whl[0]}
!pip uninstall -y torchao
!pip install -U transformers peft trl accelerate datasets flash-linear-attention

Processing /kaggle/input/datasets/erygfry/cached-data/causal_conv1d-1.7.0-cp313-cp313-linux_x86_64.whl
causal-conv1d is already installed with the same version as the provided wheel. Use --force-reinstall to force an installation of the wheel.


In [5]:
!python -c "import sys, torch; print(sys.version); print(torch.__version__, 'cuda', torch.version.cuda)"
!nvcc --version
!nproc; free -h

3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
2.11.0+cu130 cuda 13.0
nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Wed_Aug_20_01:58:59_PM_PDT_2025
Cuda compilation tools, release 13.0, V13.0.88
Build cuda_13.0.r13.0/compiler.36424714_0
4
               total        used        free      shared  buff/cache   available
Mem:            31Gi       1.7Gi        17Gi       1.7Mi        12Gi        29Gi
Swap:             0B          0B          0B


In [6]:
!pip show causal-conv1d

Name: causal_conv1d
Version: 1.7.0
Summary: Causal depthwise conv1d in CUDA, with a PyTorch interface
Home-page: https://github.com/Dao-AILab/causal-conv1d
Author: Tri Dao
Author-email: tri@tridao.me
License: 
Location: /usr/local/lib/python3.13/dist-packages
Requires: ninja, packaging, torch
Required-by: 


In [7]:
import torch, torch.nn.functional as F
from causal_conv1d import causal_conv1d_fn

B, D, T, W = 2, 64, 128, 4
x = torch.randn(B, D, T, device="cuda", dtype=torch.float16)
w = torch.randn(D, W, device="cuda", dtype=torch.float16)
b = torch.randn(D, device="cuda", dtype=torch.float16)

out = causal_conv1d_fn(x, w, b, activation="silu")
# эталон: depthwise-свёртка с паддингом слева, обрезаем хвост — так она становится причинной
ref = F.silu(F.conv1d(x.float(), w.float().unsqueeze(1), b.float(),
                      padding=W - 1, groups=D)[..., :T])
print("max diff:", (out.float() - ref).abs().max().item())

max diff: 0.003063201904296875


In [8]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.models.qwen3_5 import modeling_qwen3_5 as mq
print("fast path:", getattr(mq, "is_fast_path_available", "флага нет"))

name = "Qwen/Qwen3.5-0.8B-Base"
tok = AutoTokenizer.from_pretrained(name)
model = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float16).cuda().eval()
# в старых версиях transformers вместо dtype= было torch_dtype=

# какие реализации реально лежат внутри слоёв
seen = set()
for m in model.modules():
    for attr, val in vars(m).items():
        if callable(val) and ("conv1d" in attr or "delta_rule" in attr):
            seen.add((attr, getattr(val, "__module__", "?")))
for s in sorted(seen):
    print(s)

fast path: флага нет


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

In [9]:
# 1. Что transformers сам думает про fla
from transformers.utils import import_utils
print("fla available:", import_utils.is_flash_linear_attention_available())

# 2. Импортируется ли fla напрямую — если упадёт, трейсбек и есть ответ
from fla.ops.gated_delta_rule import chunk_gated_delta_rule, fused_recurrent_gated_delta_rule
print("fla import ok")

# 3. Что лежит в модуле модели под этими именами
for n in dir(mq):
    if any(s in n.lower() for s in ("delta", "conv1d", "fla", "fast")):
        print(n, "->", getattr(mq, n))

fla available: True
fla import ok
FlashAttentionKwargs -> <class 'transformers.modeling_flash_attention_utils.FlashAttentionKwargs'>
Qwen3_5GatedDeltaNet -> <class 'transformers.models.qwen3_5.modeling_qwen3_5.Qwen3_5GatedDeltaNet'>
causal_conv1d_fn -> <function causal_conv1d_fn at 0x7ee9fac85bc0>
causal_conv1d_update -> <function causal_conv1d_update at 0x7ee9fac85a80>
is_flash_attention_requested -> <function is_flash_attention_requested at 0x7ee9fc905da0>
torch_chunk_gated_delta_rule -> <function torch_chunk_gated_delta_rule at 0x7ee9fac85e40>
torch_recurrent_gated_delta_rule -> <function torch_recurrent_gated_delta_rule at 0x7ee9fac86020>


In [10]:
import random
text = "Фильм оставил очень сильное впечатление: прекрасная игра актёров, продуманный сюжет и отличная музыка."
words = text.split(); random.seed(0); random.shuffle(words)

for t in [text, " ".join(words)]:
    enc = tok(t, return_tensors="pt").to("cuda")
    with torch.no_grad():
        loss = model(**enc, labels=enc.input_ids).loss
    print(f"{loss.item():.3f}  {t[:50]}")

enc = tok("Отзыв: Фильм оставил", return_tensors="pt").to("cuda")
print(tok.decode(model.generate(**enc, max_new_tokens=40, do_sample=False)[0]))

2.592  Фильм оставил очень сильное впечатление: прекрасна
5.580  оставил и сюжет прекрасная отличная очень сильное 
Отзыв: Фильм оставил меня в шоке.
Отзыв: Фильм оставил меня в шоке.
Отзыв: Фильм оставил меня в шоке.
Отзыв: Фильм оставил меня


In [11]:
from torch.profiler import profile, ProfilerActivity

enc = tok(text, return_tensors="pt").to("cuda")
with torch.no_grad(), profile(activities=[ProfilerActivity.CUDA]) as prof:
    model(**enc)

names = sorted({e.key for e in prof.key_averages()})
hits = [n for n in names if any(s in n.lower() for s in ("conv1d", "delta", "chunk", "fla"))]
print(len(names), "разных ядер")
for n in hits:
    print(n[:120])

/usr/local/lib/python3.13/dist-packages/torch/profiler/profiler.py:224: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(


71 разных ядер
chunk_fwd_kernel_o
chunk_gated_delta_rule_fwd_kernel_h_blockdim64
chunk_gated_delta_rule_fwd_kkt_solve_kernel
chunk_local_cumsum_scalar_kernel
void causal_conv1d_channellast_fwd_kernel<Causal_conv1d_channellast_fwd_kernel_traits<128, 4, 64, true, c10::Half, c10::


In [12]:
import time
import torch
import pandas as pd

def bench_generate(model, tok, batch_size, new_tokens=128,
                   prompt="Отзыв: Фильм оставил", repeats=2):
    tok.padding_side = "left"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    enc = tok([prompt] * batch_size, return_tensors="pt", padding=True).to("cuda")

    gen_kwargs = dict(
        max_new_tokens=new_tokens,
        min_new_tokens=new_tokens,
        do_sample=True, temperature=1.0, top_p=1.0, top_k=0,
        pad_token_id=tok.pad_token_id,
    )

    with torch.no_grad():
        model.generate(**enc, **gen_kwargs)

    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        for _ in range(repeats):
            model.generate(**enc, **gen_kwargs)
    torch.cuda.synchronize()
    dt = (time.perf_counter() - t0) / repeats

    return {
        "batch": batch_size,
        "sec": round(dt, 2),
        "tok/s": round(batch_size * new_tokens / dt),
        "peak_GB": round(torch.cuda.max_memory_allocated() / 1e9, 2),
    }

results = [bench_generate(model, tok, bs) for bs in (1, 8, 32, 64, 128, 256)]
pd.DataFrame(results)

,batch,sec,tok/s,peak_GB
0,1,7.28,18,1.54
1,8,7.50,136,1.74
2,32,8.03,510,2.42
3,64,8.86,924,3.32
4,128,12.95,1265,5.13
5,256,24.20,1354,8.75


In [13]:
import time, torch
x = torch.randn(8192, 8192, device="cuda", dtype=torch.float16)
torch.cuda.synchronize()

t0 = time.perf_counter()
y = x @ x
t1 = time.perf_counter()          # Python уже здесь, а GPU, возможно, ещё считает
torch.cuda.synchronize()
t2 = time.perf_counter()          # а теперь GPU точно закончил

print(f"без синхронизации: {(t1 - t0) * 1000:.2f} мс")
print(f"с синхронизацией:  {(t2 - t0) * 1000:.2f} мс")

без синхронизации: 0.47 мс
с синхронизацией:  27.50 мс


In [14]:
del model
import gc
gc.collect()
torch.cuda.empty_cache()

In [15]:
import math, time, gc
import torch
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

name = "Qwen/Qwen3.5-0.8B-Base"

# --- модель с LoRA ---
base = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float16).cuda()
lora_cfg = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
                      target_modules="all-linear", task_type="CAUSAL_LM")
model = get_peft_model(base, lora_cfg)
model.print_trainable_parameters()

def train_log(model, batch_size=4, seq_len=192, steps=30):
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.enable_input_require_grads()
    model.train()
    ids = tok(text, return_tensors="pt").input_ids[:, :seq_len].repeat(batch_size, 1).cuda()
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-4)
    scaler = torch.amp.GradScaler("cuda")

    for step in range(steps):
        with torch.autocast("cuda", dtype=torch.float16):
            loss = model(input_ids=ids, labels=ids).loss
        scaler.scale(loss).backward()
        scale_before = scaler.get_scale()
        scaler.step(opt)
        scaler.update()
        skipped = scaler.get_scale() < scale_before   # множитель уменьшился → шаг пропущен
        opt.zero_grad(set_to_none=True)
        print(f"{step:2d}  loss {loss.item():.4f}  scale {scale_before:>8.0f}  {'SKIP' if skipped else ''}")

train_log(model)

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


trainable params: 10,822,656 || all params: 763,215,680 || trainable%: 1.4180
 0  loss 2.5932  scale    65536  SKIP
 1  loss 2.5932  scale    32768  SKIP
 2  loss 2.5932  scale    16384  SKIP
 3  loss 2.5932  scale     8192  SKIP
 4  loss 2.5932  scale     4096  SKIP
 5  loss 2.5932  scale     2048  SKIP
 6  loss 2.5932  scale     1024  SKIP
 7  loss 2.5932  scale      512  
 8  loss 1.5577  scale      512  
 9  loss 1.0116  scale      512  
10  loss 0.5738  scale      512  
11  loss 0.2656  scale      512  
12  loss 0.0861  scale      512  
13  loss 0.0269  scale      512  
14  loss 0.0132  scale      512  
15  loss 0.0088  scale      512  
16  loss 0.0066  scale      512  
17  loss 0.0048  scale      512  
18  loss 0.0038  scale      512  
19  loss 0.0031  scale      512  
20  loss 0.0026  scale      512  
21  loss 0.0021  scale      512  
22  loss 0.0018  scale      512  
23  loss 0.0016  scale      512  
24  loss 0.0014  scale      512  
25  loss 0.0013  scale      512  
26  loss 0

In [15]:
import math, time, gc
import torch
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

name = "Qwen/Qwen3.5-0.8B-Base"

# --- модель с LoRA ---
base = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float16).cuda()
lora_cfg = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
                      target_modules="all-linear", task_type="CAUSAL_LM")
model = get_peft_model(base, lora_cfg)
model.print_trainable_parameters()

# --- данные: настоящий русский текст, нарезанный в батч ---
text = ("Фильм оставил очень сильное впечатление: прекрасная игра актёров, "
        "продуманный сюжет и отличная музыка. ") * 40

def train_step_check(model, batch_size=8, seq_len=256, grad_ckpt=False, steps=3):
    if grad_ckpt:
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        model.enable_input_require_grads()
    else:
        model.gradient_checkpointing_disable()
    model.train()

    ids = tok(text, return_tensors="pt").input_ids[:, :seq_len].repeat(batch_size, 1).cuda()
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-4)
    scaler = torch.amp.GradScaler("cuda")

    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    losses = []
    for i in range(steps + 1):              # шаг 0 — прогрев, по нему время не считаем
        if i == 1:
            torch.cuda.synchronize()
            t0 = time.perf_counter()
        with torch.autocast("cuda", dtype=torch.float16):
            loss = model(input_ids=ids, labels=ids).loss
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        opt.zero_grad(set_to_none=True)
        losses.append(loss.item())
    torch.cuda.synchronize()

    bad = sum(int(not torch.isfinite(p).all()) for p in model.parameters() if p.requires_grad)
    return {
        "grad_ckpt": grad_ckpt,
        "losses": [round(l, 3) for l in losses],
        "all_finite": all(math.isfinite(l) for l in losses) and bad == 0,
        "sec/step": round((time.perf_counter() - t0) / steps, 2),
        "peak_GB": round(torch.cuda.max_memory_allocated() / 1e9, 2),
        "batch": batch_size,
    }


for bs in (2, 4, 8):
    for ckpt in (True, False):
        try:
            print(train_step_check(model, batch_size=bs, seq_len=192, grad_ckpt=ckpt))
        except torch.OutOfMemoryError:
            print({"batch": bs, "grad_ckpt": ckpt, "result": "OOM"})
        gc.collect()
        torch.cuda.empty_cache()

# print(train_step_check(model, grad_ckpt=False))
# print(train_step_check(model, grad_ckpt=True))

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


trainable params: 10,822,656 || all params: 763,215,680 || trainable%: 1.4180
{'grad_ckpt': True, 'losses': [0.43, 0.43, 0.43, 0.43], 'all_finite': True, 'sec/step': 0.61, 'peak_GB': 3.51, 'batch': 2}
{'grad_ckpt': False, 'losses': [0.43, 0.43, 0.43, 0.43], 'all_finite': True, 'sec/step': 0.39, 'peak_GB': 4.6, 'batch': 2}
{'grad_ckpt': True, 'losses': [0.43, 0.43, 0.43, 0.43], 'all_finite': True, 'sec/step': 0.9, 'peak_GB': 4.67, 'batch': 4}
{'grad_ckpt': False, 'losses': [0.43, 0.43, 0.43, 0.43], 'all_finite': True, 'sec/step': 0.69, 'peak_GB': 6.84, 'batch': 4}
{'grad_ckpt': True, 'losses': [0.43, 0.43, 0.43, 0.43], 'all_finite': True, 'sec/step': 1.85, 'peak_GB': 7.0, 'batch': 8}
{'grad_ckpt': False, 'losses': [0.244, 0.244, 0.244, 0.243], 'all_finite': True, 'sec/step': 1.35, 'peak_GB': 11.32, 'batch': 8}


In [16]:
from torch.profiler import profile, ProfilerActivity

# --- свежая базовая модель без LoRA ---
del model, base
gc.collect(); torch.cuda.empty_cache()
model = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float16).cuda().eval()

tok.padding_side = "left"
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
N = 30
gen = dict(max_new_tokens=N, min_new_tokens=N, do_sample=False, pad_token_id=tok.pad_token_id)

# --- 1. проверка паддинга ---
a = "Отзыв: Фильм оставил"
b = "Отзыв: Это одна из самых длинных и подробных рецензий, которые я когда-либо писал о кино, и"

with torch.no_grad():
    single = model.generate(**tok([a], return_tensors="pt").to("cuda"), **gen)
    batch  = model.generate(**tok([a, b], return_tensors="pt", padding=True).to("cuda"), **gen)

new_single, new_batch = single[0, -N:], batch[0, -N:]      # только сгенерированная часть
diff = (new_single != new_batch).nonzero()
print("совпадают первые", N if len(diff) == 0 else diff[0].item(), "токенов из", N)
print("один:  ", tok.decode(new_single))
print("батч:  ", tok.decode(new_batch))

# --- 2. какие ядра работают при генерации ---
enc = tok([a] * 8, return_tensors="pt").to("cuda")
with torch.no_grad():
    model.generate(**enc, max_new_tokens=5, pad_token_id=tok.pad_token_id)         # прогрев
    with profile(activities=[ProfilerActivity.CUDA]) as prof:
        model.generate(**enc, max_new_tokens=20, pad_token_id=tok.pad_token_id)

for n in sorted({e.key for e in prof.key_averages()}):
    if any(s in n.lower() for s in ("conv1d", "delta", "chunk", "recurrent")):
        print(n[:120])

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

совпадают первые 30 токенов из 30
один:    меня в шоке.
Отзыв: Фильм оставил меня в шоке.
Отзыв: Фильм оставил меня в шоке
батч:    меня в шоке.
Отзыв: Фильм оставил меня в шоке.
Отзыв: Фильм оставил меня в шоке
chunk_fwd_kernel_o
chunk_gated_delta_rule_fwd_kernel_h_blockdim64
chunk_gated_delta_rule_fwd_kkt_solve_kernel
chunk_local_cumsum_scalar_kernel
fused_recurrent_gated_delta_rule_fwd_kernel
void causal_conv1d_channellast_fwd_kernel<Causal_conv1d_channellast_fwd_kernel_traits<128, 4, 64, true, c10::Half, c10::
void causal_conv1d_update_kernel<Causal_conv1d_update_kernel_traits<64, 4, c10::Half, c10::Half>, false>(ConvParamsBase)
